# Shor's Algorithm — Factoring N = 15

A runnable walkthrough of the implementation in [`shor_factoring.py`](shor_factoring.py).

Shor's algorithm is **one quantum subroutine (order-finding) wrapped in a classical loop**. Below we:

1. build the order-finding circuit (quantum phase estimation),
2. read the phase peaks off the counting register,
3. decode the order `r` and recover the factors of 15,
4. note how to point the *same* code at real IBM hardware.

Everything runs on the local `AerSimulator`; the harness is backend-agnostic.

In [ ]:
from fractions import Fraction
from math import gcd

from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

from shor_factoring import (
    c_amod15,
    inverse_qft,
    build_order_finding_circuit,
    run_circuit_and_get_counts,
    find_order,
    shor,
)

backend = AerSimulator()
backend

## 1. The order-finding circuit

8 counting qubits + 4 work qubits = 12 qubits. The counting register is put in uniform
superposition, each `count[k]` controls `a^(2^k) mod 15` on the work register, then a
hand-built inverse QFT maps the accumulated phase back to a measurable integer.

In [ ]:
a = 7
qc = build_order_finding_circuit(a, n_count=8)
print(qc.num_qubits, 'qubits,', qc.size(), 'gates (pre-transpile)')
qc.draw('mpl', fold=-1)

## 2. Phase peaks

For an order-`r` base, the measured values cluster at multiples of `256 / r`.
For `a = 7` (order 4) that means clean spikes at **0, 64, 128, 192**.

In [ ]:
counts = run_circuit_and_get_counts(qc, backend, shots=2048)
decimal = {str(int(b, 2)): c for b, c in counts.items()}
plot_histogram(decimal, title=f'Counting-register readout for a={a}')

## 3. Decode the order, then factor

Continued fractions turn each measured phase `s/256` into the order `r`
(`Fraction(phase).limit_denominator(15)`). Once `r` is even and `a^(r/2) != -1 mod 15`,
the factors are `gcd(a^(r/2) +/- 1, 15)`.

In [ ]:
r, _ = find_order(a, backend)
print(f'recovered order of {a} mod 15: r = {r}')

x = pow(a, r // 2, 15)
print('factors:', gcd(x - 1, 15), 'x', gcd(x + 1, 15))

In [ ]:
# Full algorithm: random base, retry on failure.
import random
random.seed(2)
print('shor(backend) ->', shor(backend))

## 4. Per-base orders

Sweep every valid base and confirm the recovered order matches the true
multiplicative order mod 15.

In [ ]:
for a in (2, 4, 7, 8, 11, 13):
    r, _ = find_order(a, backend)
    true_r = next(k for k in range(1, 16) if pow(a, k, 15) == 1)
    status = 'ok' if r == true_r else 'MISMATCH'
    print(f'a={a:2d}  recovered r={r}  true r={true_r}  {status}')

## 5. Running on real IBM hardware

The harness is backend-agnostic — swap the backend and nothing else changes:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()           # after save_account(...) once
hw = service.least_busy(operational=True, simulator=False)
print(shor(hw))
```

Expect the phase peaks to smear on current NISQ devices: validate on the
simulator, then run on hardware and report the degradation.